# Vinamilk OCR benchmark (cloud GPU)

Use a Kaggle GPU notebook to run one OCR engine per session against the same three manually transcribed public Vinamilk report crops. In notebook settings, enable **Internet** and select a **GPU accelerator**.

This repository is private. Add a Kaggle Secret named `GITHUB_READ_TOKEN` containing a fine-grained, read-only GitHub token scoped to this repository's Contents. Never paste the token into a cell. The notebook downloads only public reports listed in the repository manifest; do not upload private enterprise documents to free notebook providers.

The three labeled crops are a smoke/evaluation fixture, not enough to select a production model. See `docs/technical/document-processing.md` for limitations and next corpus requirements. The notebook pins Surya OCR 0.17.1 (the pre-v2 recognition-predictor API) to run directly in the Kaggle GPU process. This is intentionally versioned and must be labeled as Surya v1 in reported results; Surya 2 has a different server-backed architecture.

In [ ]:
# Choose exactly one engine per notebook session.
ENGINE = "paddleocr"  # paddleocr | easyocr | surya | tesseract
REPOSITORY = "Lenhan231/enterprise-knowledge-ai-agent"
BRANCH = "feature/vietnam-enterprise-corpus"
DPI = 180
assert ENGINE in {"paddleocr", "easyocr", "surya", "tesseract"}

In [ ]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import os, subprocess

token = UserSecretsClient().get_secret("GITHUB_READ_TOKEN")
repo = Path("/kaggle/working/enterprise-knowledge-ai-agent")
askpass = Path("/kaggle/working/git-askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *Password*) echo "$GITHUB_READ_TOKEN" ;; esac\n')
askpass.chmod(0o700)
env = os.environ | {"GITHUB_READ_TOKEN": token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"}
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, f"https://github.com/{REPOSITORY}.git", str(repo)], env=env, check=True)
del token, env
askpass.unlink(missing_ok=True)
%cd /kaggle/working/enterprise-knowledge-ai-agent

In [ ]:
# Install only the selected OCR stack to avoid conflicting framework versions.
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "PyMuPDF==1.27.2.3", "Pillow==12.1.0"], check=True)
if ENGINE == "paddleocr":
    subprocess.run([sys.executable, "-m", "pip", "install", "paddlepaddle-gpu==3.3.0", "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu126/"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "paddleocr==3.7.0"], check=True)
elif ENGINE == "easyocr":
    subprocess.run([sys.executable, "-m", "pip", "install", "easyocr==1.7.2"], check=True)
elif ENGINE == "surya":
    subprocess.run([sys.executable, "-m", "pip", "install", "surya-ocr==0.17.1"], check=True)
elif ENGINE == "tesseract":
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "tesseract-ocr", "tesseract-ocr-vie"], check=True)

In [ ]:
# Keep framework downloads out of the Git checkout and record actual hardware.
os.environ["HF_HOME"] = "/kaggle/working/model-cache/huggingface"
os.environ["TORCH_HOME"] = "/kaggle/working/model-cache/torch"
subprocess.run(["nvidia-smi"], check=False)
import torch
print("torch", torch.__version__, "CUDA", torch.cuda.is_available())
if ENGINE in {"paddleocr", "easyocr"} and not torch.cuda.is_available():
    raise RuntimeError("GPU runtime was requested but PyTorch cannot see CUDA; restart with a Kaggle GPU accelerator.")
if ENGINE == "paddleocr":
    import paddle
    print("paddle", paddle.__version__, "compiled with CUDA", paddle.is_compiled_with_cuda(), "device", paddle.device.get_device())

In [ ]:
subprocess.run([sys.executable, "scripts/download_vietnam_enterprise_corpus.py", "--ids", "VNM-AR-2024-VI", "VNM-SR-2024-EN", "VNM-AR-2020-VI"], check=True)
output = Path("/kaggle/working") / f"vinamilk-ocr-{ENGINE}.json"
subprocess.run([sys.executable, "scripts/evaluate_ocr.py", "--dataset", "data/evaluation/vinamilk_ocr_ground_truth_v2.json", "--engines", ENGINE, "--dpi", str(DPI), "--output", str(output)], check=True)
print(f"Download this result from the Kaggle Output panel: {output}")